<a href="https://colab.research.google.com/github/tmedeirosb/tsi-am/blob/main/MODULO_02_pipeline_wandb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# exibir todas as colunas
pd.set_option('display.max_columns', None)

url = "https://raw.githubusercontent.com/tmedeirosb/tsi-ad-2025/refs/heads/main/dados/dados_workflow_ivan.csv"
df = pd.read_csv(url)

display(df.head())

display(df.info())

,LnguaPortuguesaeLiteraturaI90H,LnguaPortuguesaeLiteraturaI90H_dependencia,LnguaPortuguesaeLiteraturaI90H_freq,MatemticaI120H,MatemticaI120H_dependencia,MatemticaI120H_freq,aluno_exclusivo_rede_publica,descricao,descricao_area_residencial,descricao_companhia_domiciliar,descricao_estado_civil,descricao_historico,descricao_imovel,descricao_mae_escolaridade,descricao_pai_escolaridade,descricao_raca,descricao_responsavel_escolaridade,descricao_responsavel_financeiro,descricao_trabalho,id,idade,pessoa_fisica__sexo,possui_necessidade_especial,qnt_pc,qtd_pessoas_domicilio,sigla,tempo_entre_conclusao_ingresso,qnt_salarios,artificial,classe,conceito,conceito_freq,acompanhamento
0,68.0,0,100.0,66.0,0,100.0,False,Matriculado,Urbana,Mãe,Solteiro(a),Técnico de Nivel Médio em Informática,Alugado,Ensino fundamental incompleto,Ensino fundamental incompleto,Branca,Ensino fundamental incompleto,Mãe,Não informado,457884597605,15,F,False,0,2,LAJ,1,1,0,1,B,A,0
1,73.0,1,100.0,36.0,1,91.0,False,Cancelado,Urbana,Mãe,Solteiro(a),Técnico de Nível Médio em Meio Ambiente,Alugado,Ensino fundamental incompleto,Ensino fundamental incompleto,Parda,Ensino fundamental incompleto,Mãe,Não informado,458436647741,17,M,False,1,2,SPP,2,1,1,0,R,R,0
2,61.0,0,100.0,60.0,0,100.0,False,Matriculado,Urbana,Pais,Solteiro(a),Técnico de Nivel Médio em Informática,Alugado,Ensino médio completo,Ensino médio completo,Parda,Ensino médio completo,Mãe,Não informado,464029640533,15,M,True,0,5,CANG,2,2,0,1,B,A,0
3,NaN,-1,NaN,NaN,-1,NaN,False,Matriculado,Urbana,Pais,Solteiro(a),Técnico de Nível Médio em Mecânica,Alugado,Alfabetizado,Não estudou,Parda,Ensino fundamental incompleto,Mãe,Não está trabalhando,556372572373,16,F,False,1,5,MO,3,2,0,1,S,S,0
4,69.0,0,100.0,63.0,0,100.0,False,Matriculado,Urbana,Mãe,Solteiro(a),Técnico de Nivel Médio Informática,Alugado,Ensino fundamental incompleto,Ensino fundamental incompleto,Preta,Ensino fundamental incompleto,Mãe,Não está trabalhando,457614148801,15,F,False,0,3,PAR,1,1,0,1,R,A,2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8908 entries, 0 to 8907
Data columns (total 33 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   LnguaPortuguesaeLiteraturaI90H              7218 non-null   float64
 1   LnguaPortuguesaeLiteraturaI90H_dependencia  8908 non-null   int64  
 2   LnguaPortuguesaeLiteraturaI90H_freq         7215 non-null   float64
 3   MatemticaI120H                              7213 non-null   float64
 4   MatemticaI120H_dependencia                  8908 non-null   int64  
 5   MatemticaI120H_freq                         7210 non-null   float64
 6   aluno_exclusivo_rede_publica                8908 non-null   bool   
 7   descricao                                   8908 non-null   object 
 8   descricao_area_residencial                  8908 non-null   object 
 9   descricao_companhia_domiciliar              8908 non-null   object 
 10  descricao_es

None

## Pipeline de Pré-processamento e Modelagem (Versão Refatorada)

Nesta seção, utilizamos o `ColumnTransformer` para aplicar diferentes transformações em grupos específicos de colunas e o `Pipeline` para encadear essas transformações com o modelo final (`DecisionTreeClassifier`).

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Carregamento dos dados brutos (recarregando para garantir estado limpo)
url = "https://raw.githubusercontent.com/tmedeirosb/tsi-ad-2025/refs/heads/main/dados/dados_workflow_ivan.csv"
df_raw = pd.read_csv(url)

# 2. Definição das colunas por tipo de tratamento
# Colunas para Escalonamento (Numéricas)
num_features = [
    'LnguaPortuguesaeLiteraturaI90H',
    'LnguaPortuguesaeLiteraturaI90H_freq',
    'MatemticaI120H',
    'MatemticaI120H_freq'
]

# Colunas para Transformação Logarítmica
log_features = ['qnt_salarios']

# Colunas Categóricas para One-Hot Encoding
cat_features = [
    'descricao_area_residencial', 'descricao_companhia_domiciliar',
    'descricao_estado_civil', 'descricao_historico', 'descricao_imovel',
    'descricao_mae_escolaridade', 'descricao_pai_escolaridade', 'descricao_raca',
    'descricao_responsavel_escolaridade', 'descricao_responsavel_financeiro',
    'descricao_trabalho', 'pessoa_fisica__sexo', 'sigla'
]

# 3. Definição dos Pipelines de Transformação
# Pipeline para números: Imputação (Média) -> Escalonamento (StandardScaler)
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Pipeline para Log: Imputação -> Log(1+x)
log_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p))
])

# Pipeline para Categorias: One-Hot Encoding (drop_first para evitar colinearidade)
cat_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

# 4. Composição do ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('log', log_transformer, log_features),
        ('cat', cat_transformer, cat_features)
    ],
    remainder='drop' # Remove colunas não listadas (como IDs ou originais não tratadas)
)

# 5. Pipeline Final (Pré-processamento + Modelo)
clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42, max_depth=5))
])

# 6. Preparação para Treino
X = df_raw.drop(columns=['classe', 'id'])
y = df_raw['classe']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 7. Treinamento
clf_pipeline.fit(X_train, y_train)

# 8. Avaliação
y_pred = clf_pipeline.predict(X_test)

print("Relatório de Classificação:")
print(classification_report(y_test, y_pred))

# Visualizando a importância das features (opcional)
print("\nModelo treinado com sucesso utilizando ColumnTransformer e Pipeline.")

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.58      0.54      0.56       111
           1       0.97      0.97      0.97      1671

    accuracy                           0.95      1782
   macro avg       0.78      0.76      0.77      1782
weighted avg       0.95      0.95      0.95      1782


Modelo treinado com sucesso utilizando ColumnTransformer e Pipeline.


## Workflow Integrado: Pipeline + Versionamento (wandb)

Este código executa o ciclo completo de Ciência de Dados: carregamento, rastreamento de experimentos, pré-processamento via pipelines e versionamento de modelos.

In [3]:
import pandas as pd
import numpy as np
import wandb
import joblib
import os
from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

# 1. Autenticação e Inicialização
try:
    api_key = userdata.get('WANDB_API_KEY')
    os.environ["WANDB_API_KEY"] = api_key
    wandb.login()
except Exception:
    wandb.login()

run = wandb.init(
    project="modulo_02_am",
    name="integrated_pipeline_with_splits",
    job_type="end-to-end",
    config={
        "model_type": "DecisionTreeClassifier",
        "max_depth": 5,
        "test_size": 0.2,
        "random_state": 42
    }
)

# 2. Carga de Dados e Artefato Bruto
url = "https://raw.githubusercontent.com/tmedeirosb/tsi-ad-2025/refs/heads/main/dados/dados_workflow_ivan.csv"
df_raw = pd.read_csv(url)
df_raw.to_csv("raw_data.csv", index=False)

artifact_raw = wandb.Artifact(name="dataset_raw", type="dataset")
artifact_raw.add_file("raw_data.csv")
run.log_artifact(artifact_raw)

# 3. Split dos Dados
X = df_raw.drop(columns=['classe', 'id'])
y = df_raw['classe']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=run.config.test_size,
    random_state=run.config.random_state,
    stratify=y
)

# 4. Versionamento dos Splits (X_train, X_test, y_train, y_test)
splits_artifact = wandb.Artifact(name="data_splits", type="dataset")
for name, data in [("X_train", X_train), ("X_test", X_test), ("y_train", y_train), ("y_test", y_test)]:
    filename = f"{name}.csv"
    data.to_csv(filename, index=False)
    splits_artifact.add_file(filename)
run.log_artifact(splits_artifact)

# 5. Engenharia de Atributos e Pipeline
num_features = ['LnguaPortuguesaeLiteraturaI90H', 'LnguaPortuguesaeLiteraturaI90H_freq', 'MatemticaI120H', 'MatemticaI120H_freq']
log_features = ['qnt_salarios']
cat_features = ['descricao_area_residencial', 'descricao_companhia_domiciliar', 'descricao_estado_civil',
                'descricao_historico', 'descricao_imovel', 'descricao_mae_escolaridade',
                'descricao_pai_escolaridade', 'descricao_raca', 'descricao_responsavel_escolaridade',
                'descricao_responsavel_financeiro', 'descricao_trabalho', 'pessoa_fisica__sexo', 'sigla']

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())]), num_features),
    ('log', Pipeline([('imputer', SimpleImputer(strategy='median')), ('log', FunctionTransformer(np.log1p))]), log_features),
    ('cat', Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))]), cat_features)
], remainder='drop')

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=run.config.max_depth, random_state=run.config.random_state))
])

# 6. Treino e Avaliação
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
run.log({"accuracy": report['accuracy'], "f1_class_0": report['0']['f1-score'], "f1_class_1": report['1']['f1-score']})

# 7. Salvamento e Versionamento do Modelo
model_path = "full_model_pipeline.pkl"
joblib.dump(full_pipeline, model_path)
artifact_model = wandb.Artifact(name="final_pipeline_model", type="model")
artifact_model.add_file(model_path)
run.log_artifact(artifact_model)

print("Workflow completo com splits versionados e logado no W&B.")
run.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: tmedeirosb to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Workflow completo com splits versionados e logado no W&B.


accuracy,▁
f1_class_0,▁
f1_class_1,▁
accuracy,0.94725
f1_class_0,0.56075
f1_class_1,0.97194
